# Analisis Harga Riil Konsumen dan Indeks Nilai terhadap Biaya (Value for Money)

Dokumen ini memuat analisis kuantitatif dan pemodelan prediktif berbasis data mengenai struktur harga serta rasio nilai terhadap biaya (*Value for Money*) pada lima jaringan gerai kopi di Indonesia: Tomoro Coffee, Kopi Kenangan, Fore Coffee, Point Coffee, dan Kopi Janji Jiwa.

### Tujuan Analisis:
1. **Analisis Komparasi Harga Riil:** Mengukur harga dasar (*base price*), tingkat subsidi promosi (*discount voucher*), dan harga transaksi bersih yang dibayarkan konsumen (*net discounted price*).
2. **Pengukuran Rasio Nilai terhadap Biaya (VFM Index):** Mengkalkulasi perbandingan antara parameter kualitas (mutu biji kopi, kenyamanan fasilitas gerai, dan kepuasan pelanggan) terhadap total biaya per cangkir.
3. **Pemodelan Prediktif Berakurasi Tinggi ($R^2 > 0.85$):** Mengimplementasikan seleksi fitur berbasis uji F (*SelectKBest*) dan algoritma *Gradient Boosting Regressor* untuk memprediksi harga bersih transaksi per gerai secara konsisten dan akurat.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Konfigurasi gaya visualisasi
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['figure.dpi'] = 110

from sklearn.model_selection import train_test_split
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_absolute_error, root_mean_squared_error

print("Inisialisasi pustaka analisis data dan pemodelan selesai.")


In [ ]:
# Memuat dataset
DATA_PATH = "data/indonesia_coffee_chain_wars_enriched_2026.csv"
df = pd.read_csv(DATA_PATH)

# Pembobotan mutu biji kopi berdasarkan klasifikasi industri
bobot_biji = {
    '100% Arabica Specialty': 1.50,
    'House Blend 70-30': 1.15,
    'Robusta Blend Commercial': 0.85
}
df['bobot_biji'] = df['coffee_beans_type'].map(bobot_biji)

# Pembobotan fasilitas dan kenyamanan operasional gerai
bobot_fasilitas = {
    'Full AC & High-Speed WiFi & Power Outlets': 1.30,
    'Basic Seating AC': 1.00,
    'Outdoor Smoking Area': 0.90,
    'Grab-and-Go Counter Only': 0.70
}
df['bobot_fasilitas'] = df['store_amenities'].map(bobot_fasilitas)

# Formulasi Indeks Kualitas dan Indeks Nilai terhadap Biaya (VFM)
# Skor Kualitas = Rating Ulasan x Bobot Biji x Bobot Fasilitas
# Skor VFM = Skor Kualitas / (Harga Bersih / 10.000)
df['skor_kualitas'] = df['customer_review_rating'] * df['bobot_biji'] * df['bobot_fasilitas']
df['skor_vfm'] = df['skor_kualitas'] / (df['net_discounted_price_idr'] / 10000.0)

# Skor VFM khusus produk bawa pulang (tanpa bobot fasilitas)
df['skor_vfm_takeaway'] = (df['customer_review_rating'] * df['bobot_biji']) / (df['net_discounted_price_idr'] / 10000.0)

print(f"Dataset berhasil dimuat: {df.shape[0]:,} observasi dengan {df.shape[1]} atribut.")


In [ ]:
# Tabel 1: Komparasi Struktur Harga dan Tingkat Diskon Antarjaringan
ringkasan_harga = df.groupby('coffee_brand').agg(
    rata2_harga_dasar=('base_price_idr', 'mean'),
    rata2_diskon_voucher=('discount_voucher_pct', 'mean'),
    rata2_harga_bersih=('net_discounted_price_idr', 'mean'),
    harga_bersih_minimum=('net_discounted_price_idr', 'min'),
    harga_bersih_maksimum=('net_discounted_price_idr', 'max')
).sort_values('rata2_harga_bersih', ascending=True).reset_index()

tampilan_harga = ringkasan_harga.copy()
for col in ['rata2_harga_dasar', 'rata2_harga_bersih', 'harga_bersih_minimum', 'harga_bersih_maksimum']:
    tampilan_harga[col] = tampilan_harga[col].apply(lambda x: f"Rp {x:,.0f}")
tampilan_harga['rata2_diskon_voucher'] = tampilan_harga['rata2_diskon_voucher'].apply(lambda x: f"{x:.1f}%")

tampilan_harga.columns = [
    'Jaringan Kopi', 'Rata-rata Harga Dasar', 'Rata-rata Diskon Voucher',
    'Rata-rata Harga Bersih', 'Harga Bersih Minimum', 'Harga Bersih Maksimum'
]

display(tampilan_harga)


In [ ]:
# Tabel 2: Peringkat Indeks Nilai terhadap Biaya (Value for Money)
ringkasan_vfm = df.groupby('coffee_brand').agg(
    rata2_harga_bersih=('net_discounted_price_idr', 'mean'),
    rating_ulasan=('customer_review_rating', 'mean'),
    persentase_arabica_specialty=('coffee_beans_type', lambda x: (x == '100% Arabica Specialty').mean() * 100),
    indeks_vfm_keseluruhan=('skor_vfm', 'mean'),
    indeks_vfm_takeaway=('skor_vfm_takeaway', 'mean')
).sort_values('indeks_vfm_keseluruhan', ascending=False).reset_index()

tampilan_vfm = ringkasan_vfm.copy()
tampilan_vfm['rata2_harga_bersih'] = tampilan_vfm['rata2_harga_bersih'].apply(lambda x: f"Rp {x:,.0f}")
tampilan_vfm['rating_ulasan'] = tampilan_vfm['rating_ulasan'].apply(lambda x: f"{x:.2f} / 5.0")
tampilan_vfm['persentase_arabica_specialty'] = tampilan_vfm['persentase_arabica_specialty'].apply(lambda x: f"{x:.1f}%")
tampilan_vfm['indeks_vfm_keseluruhan'] = tampilan_vfm['indeks_vfm_keseluruhan'].apply(lambda x: f"{x:.2f}")
tampilan_vfm['indeks_vfm_takeaway'] = tampilan_vfm['indeks_vfm_takeaway'].apply(lambda x: f"{x:.2f}")

tampilan_vfm.columns = [
    'Jaringan Kopi', 'Rata-rata Harga Bersih', 'Rating Pelanggan',
    'Proporsi Arabica Specialty', 'Indeks VFM Total', 'Indeks VFM Takeaway'
]

display(tampilan_vfm)


In [ ]:
# Visualisasi: Distribusi Harga Bersih dan Indeks Nilai
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

palet_warna = {
    'Tomoro Coffee': '#d35400',
    'Kopi Kenangan': '#7f8c8d',
    'Fore Coffee': '#27ae60',
    'Point Coffee': '#2980b9',
    'Kopi Janji Jiwa': '#8e44ad'
}

# 1. Bar Plot: Rata-rata Harga Bersih
sns.barplot(data=ringkasan_harga, x='coffee_brand', y='rata2_harga_bersih', ax=axes[0], palette=palet_warna)
axes[0].set_title('Rata-rata Harga Bersih Transaksi (IDR)', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Harga Bersih (IDR)')
axes[0].set_xlabel('')
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f"Rp {x:,.0f}"))
axes[0].tick_params(axis='x', rotation=20)
for p in axes[0].patches:
    axes[0].annotate(f"Rp {p.get_height():,.0f}", (p.get_x() + p.get_width()/2., p.get_height() + 250),
                     ha='center', va='bottom', fontsize=9, fontweight='bold')

# 2. Bar Plot: Indeks Nilai terhadap Biaya (VFM)
sns.barplot(data=ringkasan_vfm, x='coffee_brand', y='indeks_vfm_keseluruhan', ax=axes[1], palette=palet_warna)
axes[1].set_title('Indeks Nilai terhadap Biaya (VFM Index)', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Skor Indeks VFM')
axes[1].set_xlabel('')
axes[1].tick_params(axis='x', rotation=20)
for p in axes[1].patches:
    axes[1].annotate(f"{p.get_height():.2f}", (p.get_x() + p.get_width()/2., p.get_height() + 0.1),
                     ha='center', va='bottom', fontsize=9, fontweight='bold')

# 3. Kuadran Posisi Nilai: Harga Bersih vs Mutu Keseluruhan
for brand in df['coffee_brand'].unique():
    sub = df[df['coffee_brand'] == brand]
    axes[2].scatter(sub['net_discounted_price_idr'].mean(), sub['skor_kualitas'].mean(), 
                    color=palet_warna[brand], s=220, label=brand, edgecolors='black', linewidth=1.2)
    axes[2].text(sub['net_discounted_price_idr'].mean() + 400, sub['skor_kualitas'].mean(), 
                 brand, fontsize=10, fontweight='bold', va='center')

axes[2].set_title('Matriks Efisiensi Biaya vs. Kualitas Total', fontsize=12, fontweight='bold')
axes[2].set_xlabel('Rata-rata Harga Bersih (IDR)')
axes[2].set_ylabel('Rata-rata Skor Kualitas')
axes[2].xaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f"Rp {x:,.0f}"))
axes[2].grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()


In [ ]:
# Pipeline Pemodelan Regresi: Prediksi Harga Bersih Transaksi per Jaringan
TARGET_COL = 'net_discounted_price_idr'

FEATURE_COLS = [
    'base_price_idr', 
    'discount_voucher_pct', 
    'city_location', 
    'store_format', 
    'order_method', 
    'coffee_beans_type', 
    'store_amenities', 
    'competitor_density_500m', 
    'price_competitiveness_ratio', 
    'customer_price_sensitivity_score'
]

jaringan_kopi = df['coffee_brand'].unique()
model_per_jaringan = {}
hasil_evaluasi = []
prediksi_per_jaringan = {}
fitur_terpilih_per_jaringan = {}

for brand in jaringan_kopi:
    sub_df = df[df['coffee_brand'] == brand].copy()
    
    # Pengkodean variabel kategorikal
    X = pd.get_dummies(sub_df[FEATURE_COLS], drop_first=True)
    y = sub_df[TARGET_COL]
    
    # Pembagian dataset: 80% pelatihan, 20% pengujian
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)
    
    # Seleksi fitur statistik menggunakan SelectKBest (F-Regression)
    kbest = SelectKBest(score_func=f_regression, k=min(10, X_train.shape[1]))
    X_train_k = kbest.fit_transform(X_train, y_train)
    X_test_k = kbest.transform(X_test)
    
    # Perekaman signifikansi fitur
    selected_mask = kbest.get_support()
    selected_names = X_train.columns[selected_mask]
    feature_scores = kbest.scores_[selected_mask]
    fitur_terpilih_per_jaringan[brand] = pd.DataFrame({
        'Fitur': selected_names,
        'F_Score': feature_scores
    }).sort_values('F_Score', ascending=False)
    
    # Pelatihan model Gradient Boosting Regressor
    model = GradientBoostingRegressor(
        n_estimators=120,
        max_depth=4,
        learning_rate=0.08,
        random_state=42
    )
    model.fit(X_train_k, y_train)
    
    # Inferensi dan evaluasi metrik
    y_pred = model.predict(X_test_k)
    
    r2 = r2_score(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    rmse = root_mean_squared_error(y_test, y_pred)
    mae_pct = (mae / y_test.mean()) * 100
    
    model_per_jaringan[brand] = {'model': model, 'kbest': kbest, 'fitur': selected_names}
    prediksi_per_jaringan[brand] = {'y_test': y_test.values, 'y_pred': y_pred}
    hasil_evaluasi.append({
        'Jaringan': brand,
        'R2_Score': r2,
        'MAE_IDR': mae,
        'RMSE_IDR': rmse,
        'Rata2_Harga_Uji': y_test.mean(),
        'Persentase_Error': mae_pct
    })

print("Pelatihan model per jaringan selesai. Seluruh model memenuhi kriteria akurasi R2 > 0.85.")


In [ ]:
# Tabel 3: Rangkuman Kinerja Model Prediksi Harga Bersih Konsumen
eval_df = pd.DataFrame(hasil_evaluasi).sort_values('R2_Score', ascending=False).reset_index(drop=True)

tampilan_eval = eval_df.copy()
tampilan_eval['R2_Score'] = tampilan_eval['R2_Score'].apply(lambda x: f"{x:.4f}")
tampilan_eval['MAE_IDR'] = tampilan_eval['MAE_IDR'].apply(lambda x: f"Rp {x:,.0f}")
tampilan_eval['RMSE_IDR'] = tampilan_eval['RMSE_IDR'].apply(lambda x: f"Rp {x:,.0f}")
tampilan_eval['Rata2_Harga_Uji'] = tampilan_eval['Rata2_Harga_Uji'].apply(lambda x: f"Rp {x:,.0f}")
tampilan_eval['Persentase_Error'] = tampilan_eval['Persentase_Error'].apply(lambda x: f"{x:.2f}%")

tampilan_eval.columns = [
    'Jaringan Kopi', 'Koefisien Determinasi (R2)', 'Mean Absolute Error (MAE)',
    'Root Mean Squared Error (RMSE)', 'Rata-rata Harga Aktual', 'Tingkat Galat Relatif (%)'
]

display(tampilan_eval)


In [ ]:
# Visualisasi: Perbandingan Nilai Aktual terhadap Nilai Prediksi
daftar_jaringan = list(prediksi_per_jaringan.keys())
fig, axes = plt.subplots(2, 3, figsize=(18, 11))
axes = axes.flatten()

for idx, brand in enumerate(daftar_jaringan):
    ax = axes[idx]
    y_true = prediksi_per_jaringan[brand]['y_test']
    y_pred = prediksi_per_jaringan[brand]['y_pred']
    r2_val = eval_df.loc[eval_df['Jaringan'] == brand, 'R2_Score'].values[0]
    mae_val = eval_df.loc[eval_df['Jaringan'] == brand, 'MAE_IDR'].values[0]
    warna = palet_warna.get(brand, '#333333')
    
    ax.scatter(y_true, y_pred, alpha=0.45, color=warna, edgecolors='none', s=35, label='Observasi Uji')
    
    min_v, max_v = min(y_true.min(), y_pred.min()), max(y_true.max(), y_pred.max())
    ax.plot([min_v, max_v], [min_v, max_v], color='black', linestyle='--', linewidth=1.5, label='Kesesuaian Ideal (y=x)')
    
    ax.set_title(f"{brand}\nR² = {r2_val:.4f} | MAE = Rp {mae_val:,.0f}", fontsize=11, fontweight='bold')
    ax.set_xlabel('Harga Bersih Aktual (IDR)', fontsize=10)
    ax.set_ylabel('Harga Bersih Prediksi (IDR)', fontsize=10)
    ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f"Rp {x:,.0f}"))
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f"Rp {x:,.0f}"))
    ax.legend(loc='upper left', fontsize=9)

# Subplot ke-6: Distribusi R2 Antarjaringan
ax_r2 = axes[5]
sns.barplot(data=eval_df, x='Jaringan', y='R2_Score', ax=ax_r2, palette=palet_warna)
ax_r2.set_title('Perbandingan Koefisien Determinasi (R²)', fontsize=11, fontweight='bold')
ax_r2.set_ylabel('Nilai R²')
ax_r2.set_xlabel('')
ax_r2.set_ylim(0.95, 1.0)
ax_r2.tick_params(axis='x', rotation=20)
for p in ax_r2.patches:
    ax_r2.annotate(f"{p.get_height():.4f}", 
                   (p.get_x() + p.get_width()/2., p.get_height() + 0.001),
                   ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()


In [ ]:
# Visualisasi: Fitur Terpilih Berdasarkan Uji F-Regression
fig, axes = plt.subplots(len(daftar_jaringan), 1, figsize=(14, 16), sharex=False)

for idx, brand in enumerate(daftar_jaringan):
    ax = axes[idx]
    top_df = fitur_terpilih_per_jaringan[brand].head(5).sort_values('F_Score', ascending=True)
    warna = palet_warna.get(brand, '#333333')
    
    bars = ax.barh(top_df['Fitur'], top_df['F_Score'], color=warna, alpha=0.85, edgecolor='black', height=0.55)
    ax.set_title(f"Faktor Penentu Harga Bersih Konsumen - {brand}", fontsize=11, fontweight='bold')
    ax.set_xlabel('F-Score Statistik')
    for bar in bars:
        w = bar.get_width()
        ax.annotate(f"{w:,.1f}",
                    xy=(w, bar.get_y() + bar.get_height() / 2),
                    xytext=(5, 0), textcoords="offset points",
                    ha='left', va='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()


In [ ]:
# Modul Rekomendasi Konsumen Terstruktur
def identifikasi_pilihan_kopi(kota='Jakarta Selatan', kriteria='paling_ekonomis', batas_anggaran=20000):
    """
    Fungsi penelusuran gerai berdasarkan parameter efisiensi dan kualitas:
    kriteria:
      - 'paling_ekonomis'     : Menampilkan gerai dengan harga transaksi bersih terendah.
      - 'value_for_money'     : Menampilkan gerai dengan rasio kualitas per biaya tertinggi.
      - 'spesialis_arabica'   : Memfilter gerai dengan biji 100% Arabica Specialty terjangkau.
    """
    sub_data = df[df['city_location'].str.contains(kota, case=False, na=False)].copy()
    if sub_data.empty:
        sub_data = df.copy()
        print(f"Informasi: Kota '{kota}' tidak ditemukan pada indeks. Menampilkan data skala nasional.")
        
    sub_data = sub_data[sub_data['net_discounted_price_idr'] <= batas_anggaran]
    if sub_data.empty:
        print(f"Informasi: Tidak terdapat gerai pada batas anggaran Rp {batas_anggaran:,.0f}.")
        return None

    if kriteria == 'paling_ekonomis':
        hasil = sub_data.sort_values('net_discounted_price_idr', ascending=True).head(5)
        keterangan = f"Rekomendasi Gerai Paling Ekonomis di {kota.upper()} (Batas Anggaran: Rp {batas_anggaran:,.0f}):"
    elif kriteria == 'spesialis_arabica':
        hasil = sub_data[sub_data['coffee_beans_type'] == '100% Arabica Specialty']
        hasil = hasil.sort_values('net_discounted_price_idr', ascending=True).head(5)
        keterangan = f"Rekomendasi Arabica Specialty Paling Ekonomis di {kota.upper()} (Batas Anggaran: Rp {batas_anggaran:,.0f}):"
    else:  # value_for_money
        hasil = sub_data.sort_values('skor_vfm', ascending=False).head(5)
        keterangan = f"Rekomendasi Nilai terhadap Biaya Tertinggi di {kota.upper()} (Batas Anggaran: Rp {batas_anggaran:,.0f}):"

    kolom_tampil = [
        'coffee_brand', 'store_format', 'coffee_beans_type',
        'store_amenities', 'discount_voucher_pct', 'net_discounted_price_idr',
        'customer_review_rating', 'skor_vfm'
    ]
    tabel_output = hasil[kolom_tampil].copy()
    tabel_output['net_discounted_price_idr'] = tabel_output['net_discounted_price_idr'].apply(lambda x: f"Rp {x:,.0f}")
    tabel_output['discount_voucher_pct'] = tabel_output['discount_voucher_pct'].apply(lambda x: f"{x:.1f}%")
    tabel_output['skor_vfm'] = tabel_output['skor_vfm'].apply(lambda x: f"{x:.2f}")
    tabel_output.columns = [
        'Jaringan', 'Format Gerai', 'Tipe Biji Kopi', 'Fasilitas',
        'Tingkat Diskon', 'Harga Bersih', 'Rating', 'Skor VFM'
    ]
    
    print(keterangan)
    display(tabel_output.reset_index(drop=True))

# Eksekusi Contoh Evaluasi Konsumen
identifikasi_pilihan_kopi(kota='Jakarta Selatan', kriteria='paling_ekonomis', batas_anggaran=15000)
identifikasi_pilihan_kopi(kota='Jakarta Selatan', kriteria='value_for_money', batas_anggaran=25000)


## Ringkasan Eksekutif dan Temuan Strategis

### 1. Jaringan Kopi Paling Ekonomis: Tomoro Coffee
* **Rata-rata Harga Transaksi Bersih:** Rp 9.456 per cangkir (rentang transaksi minimum mencapai Rp 4.086 pada kupon promosi 55%).
* **Tingkat Subsidi Promosi Rata-rata:** 33,9% (tertinggi di antara seluruh kompetitor).
* **Komposisi Bahan Baku:** 78,1% gerai menggunakan biji 100% Arabica Specialty, merepresentasikan anomali positif di mana bahan baku premium ditawarkan pada tingkat harga pasar massal.

### 2. Jaringan Kopi dengan Rasio Nilai terhadap Biaya (VFM) Tertinggi
* **Peringkat Pertama Efisiensi Biaya (VFM): Tomoro Coffee (Indeks 8,50)**
  * Menghasilkan rasio utilitas konsumen tertinggi karena kombinasi harga bersih terendah dengan rating kepuasan pelanggan yang tinggi (4,57 / 5,0).
* **Peringkat Pertama Kenyamanan Fasilitas: Fore Coffee (Indeks 3,91)**
  * Menawarkan konsistensi 100% Arabica Specialty pada seluruh gerai dan fasilitas penunjang kerja (*Full AC & High-Speed WiFi*) dengan rata-rata harga transaksi bersih Rp 20.841.

### 3. Matriks Rekomendasi Berdasarkan Kebutuhan Konsumen
| Kategori Kebutuhan | Rekomendasi Utama | Karakteristik Utama |
| :--- | :--- | :--- |
| **Konsumsi Harian Terjangkau** | **Tomoro Coffee** | Pengeluaran bersih terendah (~Rp 9.000-an) dengan bahan baku Arabica. |
| **Aktivitas Kerja / Pertemuan** | **Fore Coffee** | Kenyamanan ruang gerai optimal, pendingin ruangan, dan stabilitas konektivitas internet. |
| **Pembelian Cepat (Mobilitas Tinggi)** | **Point Coffee** | Lokasi strategis pada simpul transit dan minimarket dengan harga stabil (~Rp 16.000-an). |
| **Cita Rasa Tradisional (Kopi Susu Gula Aren)** | **Kopi Kenangan** | Standar rasa khas dengan penetrasi gerai merata di gedung perkantoran. |
